## Example 1: URI template expansion and processing

In [ ]:
import uri_template

# Example 1: URI template expansion and processing
print("Example 1: URI template expansion and processing")

try:
    # Test basic URI template expansion
    test_cases = [
        ('https://api.example.com/users/{id}', {'id': '123'}),
        ('https://api.example.com/search{?q,limit}', {'q': 'python', 'limit': 10}),
        ('https://api.example.com/users/{id}/posts{?page,size}', {'id': '456', 'page': 2, 'size': 25}),
        ('/path{/segments*}', {'segments': ['a', 'b', 'c']}),
    ]
    
    print("URI template expansion:")
    print("=" * 60)
    
    for template_str, values in test_cases:
        try:
            # Create template object
            template = uri_template.URITemplate(template_str)
            print(f"Template: {template_str}")
            print(f"Values:   {values}")
            
            # Expand using keyword arguments
            result = template.expand(**values)
            print(f"Result:   {result}")
            print(f"Variables: {template.variable_names}")
            print()
        except Exception as e:
            print(f"Error expanding {template_str}: {e}")
            print()
            raise
    
    # Show template capabilities
    complex_template = uri_template.URITemplate('https://api.example.com{/path*}{?query*}')
    print(f"Complex template: {complex_template}")
    print(f"Variables: {complex_template.variables}")
    
    result = complex_template.expand(path=['api', 'v1', 'users'], query={'limit': 10, 'sort': 'name'})
    print(f"Expanded: {result}")
    assert result == 'https://api.example.com/api/v1/users?limit=10&sort=name', result
    
    print("uri_template example 1 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"uri_template example 1 failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: Query and path-segment expansion

In [ ]:
import uri_template

# Example 2: Query and path-segment expansion
# {?q,limit} builds a form-style query string; {/segments*} explodes a list into path segments.
search = uri_template.URITemplate('https://api.example.com/search{?q,limit}')
search_uri = search.expand(q='python', limit=10)
print(f"{search} -> {search_uri}")
assert search_uri == 'https://api.example.com/search?q=python&limit=10', search_uri

# Values are percent-encoded, and a variable with no value is left out.
assert search.expand(q='a b', limit=5) == 'https://api.example.com/search?q=a%20b&limit=5'
assert search.expand(q='python') == 'https://api.example.com/search?q=python'

# The module-level expand() takes the template as a string.
assert uri_template.expand('{?q,limit}', q='a b', limit=5) == '?q=a%20b&limit=5'

segments = uri_template.URITemplate('/path{/segments*}')
segments_uri = segments.expand(segments=['a', 'b', 'c'])
print(f"{segments} -> {segments_uri}")
assert segments_uri == '/path/a/b/c', segments_uri

## Example 3: Reserved and fragment expansion

In [ ]:
import uri_template

# Example 3: Reserved and fragment expansion
# Simple {path} expansion percent-encodes reserved characters such as '/';
# {+path} keeps them, and {#section} keeps them after a leading '#'.
simple = uri_template.URITemplate('{path}/here').expand(path='/foo/bar')
reserved = uri_template.URITemplate('{+path}/here').expand(path='/foo/bar')
print(f"simple:   {simple}")
print(f"reserved: {reserved}")
assert simple == '%2Ffoo%2Fbar/here', simple
assert reserved == '/foo/bar/here', reserved

fragment = uri_template.URITemplate('https://docs.example.com/page{#section}').expand(section='install/linux')
print(f"fragment: {fragment}")
assert fragment == 'https://docs.example.com/page#install/linux', fragment

## Example 4: Template variables and the prefix modifier

In [ ]:
import uri_template

# Example 4: Template variables and the prefix modifier
template = uri_template.URITemplate('https://api.example.com/{version}/users/{user_id}{/path*}{?limit,offset}')
print(f"variable_names: {template.variable_names}")
assert set(template.variable_names) == {'version', 'user_id', 'path', 'limit', 'offset'}, template.variable_names

uri = template.expand(version='v1', user_id='1001', path=['posts', '42'], limit=5)
print(f"expanded: {uri}")
assert uri == 'https://api.example.com/v1/users/1001/posts/42?limit=5', uri

# {name:3} keeps only the first three characters of the value.
initials = uri_template.URITemplate('/initials/{name:3}')
assert initials.variable_names == ['name'], initials.variable_names
assert initials.expand(name='alexander') == '/initials/ale'

## Example 5: Template validation

In [ ]:
import uri_template

# Example 5: Template validation
# validate() reports whether a string is a well-formed RFC 6570 template.
# Building a URITemplate from a malformed one raises ExpansionInvalidError.
assert uri_template.validate('https://api.example.com/users/{id}')
assert not uri_template.validate('{')
assert not uri_template.validate('/users/{id')

try:
    uri_template.URITemplate('{')
except uri_template.ExpansionInvalidError as e:
    print(f"URITemplate('{{') raised {type(e).__name__}: {e}")
else:
    raise AssertionError("URITemplate('{') should raise ExpansionInvalidError")